In [3]:
#! pip install prometheus-api-client pandas matplotlib

In [ ]:
import pandas as pd
from prometheus_api_client import PrometheusConnect
import matplotlib.pyplot as plt

# 1. Connect to Prometheus
# Use 'http://localhost:9090' if running locally
# Use 'http://prometheus:9090' if inside the same Docker network
prom = PrometheusConnect(url="http://localhost:9090", disable_ssl=True)


+ In Prometheus, the time duration in brackets—the range vector selector—defines the look-back window that the rate() function uses to calculate the per-second average. So, for `rate(messages_consumed_total[**60m**])`, the **60m** indicates the following:
    - The rate() function calculates how much the counter `(total messages)` increased between the start and the end of that **60-minute** window, then divides that increase by the number of seconds in an hour ($3,600$ seconds).

In [226]:
# Define our targets and their business explanations
metric_configs = [
    {"q": 0.50, "name": "P50 (Median)", "desc": "The typical latency. 50% of messages are processed faster than this."},
    {"q": 0.95, "name": "P95 (Tail)", "desc": "The 'slow' case. Standard for measuring system performance and stability."},
    {"q": 0.99, "name": "P99 (Extreme)", "desc": "The worst-case scenario. Identifies rare but severe delays or bottlenecks."},
]

table_data = []

for item in metric_configs:
    query = f'histogram_quantile({item["q"]}, sum(rate(message_processing_latency_seconds_bucket[5m])) by (le))'
    result = prom.custom_query(query=query)
    
    if result:
        # We take the first result (aggregated across partitions)
        value = float(result[0]['value'][1])
        table_data.append({
            "Metric": item["name"],
            "Value (Seconds)": f"{value:.4f}",
            "Explanation": item["desc"]
        })

# 3. Create and display the DataFrame
summary_df = pd.DataFrame(table_data)

print("--- Latency Performance Summary ---")
display(summary_df)

--- Latency Performance Summary ---


,Metric,Value (Seconds),Explanation
0,P50 (Median),4.7892,The typical latency. 50% of messages are proce...
1,P95 (Tail),54.4714,The 'slow' case. Standard for measuring system...
2,P99 (Extreme),90.8943,The worst-case scenario. Identifies rare but s...


In [ ]:
metrics = {
    "Total End-to-End": "message_processing_latency_seconds_bucket",
    "Consumer Queue Time": "message_queue_time_seconds_bucket"
}
quantiles = [0.50, 0.95, 0.99]
results = []

for name, metric in metrics.items():
    row = {"Stage": name}
    for q in quantiles:
        query = f'histogram_quantile({q}, sum(rate({metric}[5m])) by (le))'
        res = prom.custom_query(query)
        val = float(res[0]['value'][1]) if res else 0
        row[f"P{int(q*100)}"] = round(val, 4)
    results.append(row)

df = pd.DataFrame(results).set_index("Stage")

# Calculate Internal
internal = (df.loc["Total End-to-End"] - df.loc["Consumer Queue Time"]).clip(lower=0)
internal.name = "Producer Internal"

# Combine and reorder
final_df = pd.concat([df, pd.DataFrame([internal])])
# This line ensures 'Total' is always at the bottom
final_df = final_df.reindex(["Producer Internal", "Consumer Queue Time", "Total End-to-End"]).reset_index()

print("Latency Summary (Seconds):")
display(final_df)

Latency Summary (Seconds):


,index,P50,P95,P99
0,Producer Internal,0.8133,0.6547,0.1310
1,Consumer Queue Time,4.1034,15.4167,19.0833
2,Total End-to-End,4.9167,16.0714,19.2143


+ Avg Msg Sent/per sec (60m): Measures the average rate at which the producer is publishing new messages to Kafka over the last hour.

+ Avg Msg Consumed/per sec (60m): Tracks the average throughput of your consumer group, showing how many messages are being successfully processed per second.

+ Active Consumers: Displays the current count of unique consumer instances connected to the system, helping verify if your consumer scaling is working as expected.

+ System Throughput: Calculated via the Prometheus Histogram count, this represents the total volume of processed events to measure the overall capacity of the pipeline.

P95 Latency (sec): Indicates the 95th percentile of end-to-end processing time; it means 95% of messages were processed within this duration, highlighting the "worst-case" performance for most users.

In [411]:
import pandas as pd

# --- 1. Get the list of Partitions dynamically ---
partition_info = prom.custom_query(query='count(count by (partition) (messages_produced_total))')
num_partitions = int(partition_info[0]['value'][1]) if partition_info else 0
partition_list = list(range(num_partitions))

# --- 2. Define Separate Query Sets ---
# These keys must match so we can pair them in the loop
partition_queries = {
    'Avg Msg Sent/per sec (60m)': 'sum(rate(messages_produced_total[60m])) by (partition)',
    'Avg Msg Consumed/per sec (60m)': 'sum(rate(messages_consumed_total[60m])) by (partition)',
    'Active Consumers': 'count(messages_consumed_total) by (partition)',
    'System Throughput': 'sum(rate(message_processing_latency_seconds_count[60m])) by (partition)',
    'P95 Latency (sec)': 'histogram_quantile(0.95, sum(rate(message_processing_latency_seconds_bucket[60m])) by (le, partition))'
}

all_queries = {
    'Avg Msg Sent/per sec (60m)': 'sum(rate(messages_produced_total[60m]))',
    'Avg Msg Consumed/per sec (60m)': 'sum(rate(messages_consumed_total[60m]))',
    'Active Consumers': 'count(count by (instance) (messages_consumed_total))',
    'System Throughput': 'sum(rate(message_processing_latency_seconds_count[60m]))',
    'P95 Latency (sec)': 'histogram_quantile(0.95, sum(rate(message_processing_latency_seconds_bucket[60m])) by (le))'
}

rows = []

# --- 3. Fetch Data ---
for label in all_queries.keys():
    metric_row = {'Metric': label}
    
    try:
        # A. Fetch the 'all' value using the specific global query
        res_all = prom.custom_query(query=all_queries[label])
        metric_row['P_all'] = round(float(res_all[0]['value'][1]), 5) if res_all else 0.0
        
        # B. Fetch the partition-specific values
        res_part = prom.custom_query(query=partition_queries[label])
        partition_map = {
            res['metric'].get('partition', 'unknown'): round(float(res['value'][1]), 5)
            for res in res_part
        }
        
        # C. Map individual partitions to columns
        for p in partition_list:
            metric_row[f"P_{p}"] = partition_map.get(str(p), 0.0)
            
    except Exception as e:
        print(f"Error fetching {label}: {e}")
        metric_row['P_all'] = "Error"
        for p in partition_list:
            metric_row[f"P_{p}"] = "Error"
            
    rows.append(metric_row)

# --- 4. Create and Display ---
summary_df = pd.DataFrame(rows)

print(f"\n=== System Overview: {num_partitions} Partitions Detected ===")

# Apply rounding to all numeric columns (columns after 'Metric')
numeric_cols = summary_df.columns.drop('Metric')
summary_df[numeric_cols] = summary_df[numeric_cols].round(3)
display(summary_df)



# display(summary_df.style.hide(axis='index').set_properties(**{'text-align': 'center'}))


=== System Overview: 10 Partitions Detected ===


,Metric,P_all,P_0,P_1,P_2,P_3,P_4,P_5,P_6,P_7,P_8,P_9
0,Avg Msg Sent/per sec (60m),0.145,0.011,0.017,0.015,0.014,0.015,0.014,0.020,0.014,0.016,0.010
1,Avg Msg Consumed/per sec (60m),0.139,0.010,0.017,0.016,0.014,0.016,0.011,0.018,0.013,0.015,0.010
2,Active Consumers,2.000,1.000,1.000,1.000,1.000,1.000,2.000,2.000,2.000,2.000,2.000
3,System Throughput,0.139,0.010,0.017,0.016,0.014,0.016,0.011,0.018,0.013,0.015,0.010
4,P95 Latency (sec),51.986,17.875,19.208,18.472,17.958,45.000,62.017,18.948,19.017,91.978,70.099


In [310]:
for res in results:
    print(res)
    print(res['metric'].get('partition', 'unknown'))
    print(round(float(res['value'][1]),5))
    print()

{'metric': {'partition': '0'}, 'value': [1773746148.611, '0.40001454598349034']}
0
0.40001

{'metric': {'partition': '5'}, 'value': [1773746148.611, '0.236372231717517']}
5
0.23637

{'metric': {'partition': '8'}, 'value': [1773746148.611, '0.236372231717517']}
8
0.23637

{'metric': {'partition': '2'}, 'value': [1773746148.611, '0.3272846285319466']}
2
0.32728

{'metric': {'partition': '9'}, 'value': [1773746148.611, '0.05454743808865777']}
9
0.05455

{'metric': {'partition': '4'}, 'value': [1773746148.611, '0.25455471108040295']}
4
0.25455

{'metric': {'partition': '1'}, 'value': [1773746148.611, '0.25455471108040295']}
1
0.25455

{'metric': {'partition': '6'}, 'value': [1773746148.611, '0.27273719044328887']}
6
0.27274

{'metric': {'partition': '7'}, 'value': [1773746148.611, '0.236372231717517']}
7
0.23637

{'metric': {'partition': '3'}, 'value': [1773746148.611, '0.1454598349030874']}
3
0.14546



In [289]:
int(partition_info[0]['value'][1])

10

In [292]:
partitions = range(1, int(partition_info[0]['value'][1]) + 1)
partitions = list(partitions)
partitions.append('all')
partitions

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 'all']

In [ ]:
partitions = sorted([series['metric'].get('partition', 'all') for series in int(partition_info[0]['value'][1])])
partitions

TypeError: string indices must be integers

In [246]:
import plotly.express as px
from datetime import datetime, timedelta
minutes_interval = 10.0
time_bucket_for_average = 5 #minutes

# 2. Setup Time Range (Last 12 hours)
end_time = datetime.now()
start_time = end_time - timedelta(hours=12)
step = "1m"

query = f'histogram_quantile(0.95, sum(rate(message_processing_latency_seconds_bucket[{time_bucket_for_average}m])) by (le, partition))'

# 3. Fetch Data
result = prom.custom_query_range(
    query=query,
    start_time=start_time,
    end_time=end_time,
    step=step,
)

# 4. Flatten the result into a DataFrame
df_list = []
for series in result:
    partition = series['metric'].get('partition', 'all')
    for val in series['values']:
        df_list.append({
            'timestamp': pd.to_datetime(val[0], unit='s'),
            'partition': f"Partition {partition}",
            'latency_sec': float(val[1])
        })

df = pd.DataFrame(df_list)
df['timestamp'] = pd.to_datetime(df['timestamp']) 

# 5. Create Plotly Graph
fig = px.line(
    df, 
    x='timestamp', 
    y='latency_sec', 
    color='partition',
    title='P95 End-to-End Latency (Interactive)',
    labels={'timestamp': 'Time', 'latency_sec': 'Latency (Seconds)'},
    template='plotly_white'
)

# Convert to milliseconds for Plotly
ms_interval = minutes_interval * 60 * 1000
# Logic to determine the best format
if minutes_interval < 60:
    # Under 1 hour: Show minutes and seconds
    custom_format = "%H:%M:%S"
elif minutes_interval < 1440: # 24 hours
    # 1 to 24 hours: Show Hour:Minute and Short Date
    custom_format = "%H:%M\n%b %d"
else:
    # Over 24 hours: Focus on the Date
    custom_format = "%b %d\n%Y"

fig.update_xaxes(
    type='date',             # Force the axis to recognize dates
    tickmode='linear',
    tick0=df['timestamp'].min(), 
    dtick=ms_interval,
    tickformat=custom_format,
    ticks="outside",
    ticklen=10,
    tickcolor='black',
    showgrid=True,
    gridcolor='LightPink'
)
fig.show()

# 6. Display Stats Table
print("--- Performance Stats ---")
stats_table = df.groupby('partition')['latency_sec'].agg(['mean', 'max', 'std']).reset_index()
display(stats_table)

--- Performance Stats ---


,partition,mean,max,std
0,Partition all,103.3393,178.333333,55.389542


In [ ]:
import pandas as pd
from prometheus_api_client import PrometheusConnect
import matplotlib.pyplot as plt

# 1. Connect to Prometheus
# Use 'http://localhost:9090' if running locally
# Use 'http://prometheus:9090' if inside the same Docker network
prom = PrometheusConnect(url="http://localhost:9090", disable_ssl=True)

# 2. Define the Query (P95 Latency)
query = 'histogram_quantile(0.95, sum(rate(message_processing_latency_seconds_bucket[5m])) by (le))'

# 3. Fetch Data as a list of dicts and convert to DataFrame
result = prom.custom_query(query=query)
df_list = []

for series in result:
    partition = series['metric'].get('partition', 'all')
    value = float(series['value'][1])
    timestamp = pd.to_datetime(series['value'][0], unit='s')
    df_list.append({'timestamp': timestamp, 'partition': partition, 'latency_sec': value})

df = pd.DataFrame(df_list)

# 4. Display the Table of Stats
print("--- Latency Statistics per Partition ---")
stats_table = df.groupby('partition')['latency_sec'].agg(['mean', 'max', 'min', 'std']).reset_index()
display(stats_table)

# 5. Simple Plot
df.pivot(index='timestamp', columns='partition', values='latency_sec').plot(kind='line', figsize=(10, 5))
plt.title("P95 Latency over Time")
plt.ylabel("Seconds")
plt.show()